# M12 · Inferência eficiente e LLMOps: prática

1. Medir: o que custa gerar com transformers, token a token.
2. Quantizar e comparar memória, velocidade e qualidade.
3. Servir com vLLM: débito com muitos pedidos em simultâneo.
4. A API de produção e o teste de carga.
5. Monitorizar: o painel e um alerta simples de drift.

**Requisitos:** GPU NVIDIA (uma T4 chega para as secções 1, 2 e 4 com o modelo de 1,5B; o vLLM precisa de uma GPU suportada), o modelo do módulo 11 em `../11-fine-tuning-e-alinhamento/use-case/modelo_final`, `uv sync --extra deploy --extra finetuning`.

Lição de apoio: [licao.html](licao.html)

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
import torch

UC = Path("use-case").resolve()
M11 = (UC.parent.parent / "11-fine-tuning-e-alinhamento" / "use-case").resolve()
MODELO = M11 / "modelo_final"
sys.path.insert(0, str(M11))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")
print("Modelo do M11:", "encontrado" if MODELO.exists() else "em falta: corre primeiro o módulo 11 (fundir.py)")

def correr(*args, fundo=False):
    if fundo:
        return subprocess.Popen([sys.executable, *args], cwd=UC, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    p = subprocess.run([sys.executable, *args], cwd=UC, capture_output=True, text=True)
    print(p.stdout[-4000:], p.stderr[-2000:] if p.returncode else "")

## 1. Prefill e decode com transformers

Medimos o tempo até ao primeiro token (prefill) e o tempo por token (decode), com um e com vários pedidos em lote.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from comum import exemplo_chat

tok = AutoTokenizer.from_pretrained(MODELO)
tok.padding_side = "left"
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float16, device_map="cuda")
texto = tok.apply_chat_template(exemplo_chat("A encomenda 51234 nunca chegou e já passaram 3 semanas! Quero cancelar.")["prompt"],
                                tokenize=False, add_generation_prompt=True)

def medir(lote, novos=64):
    enc = tok([texto] * lote, return_tensors="pt", padding=True).to("cuda")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    modelo.generate(**enc, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.perf_counter() - t0
    t0 = time.perf_counter()
    modelo.generate(**enc, max_new_tokens=novos, min_new_tokens=novos, do_sample=False)
    torch.cuda.synchronize(); total = time.perf_counter() - t0
    return ttft, (total - ttft) / (novos - 1), lote * novos / total

for lote in [1, 4, 16, 32]:
    ttft, tpt, tps = medir(lote)
    print(f"lote {lote:>2}: TTFT {ttft * 1000:6.1f} ms · {tpt * 1000:5.1f} ms/token · {tps:7.0f} tokens/s no total")

Repara que o tempo por token quase não sobe com o lote, mas os tokens por segundo multiplicam-se: o decode está limitado pela leitura dos pesos, e um lote aproveita cada leitura. É esta a razão de ser do continuous batching.

**Exercício 1.1.** Calcula o KV cache por token deste modelo (`modelo.config`: camadas, `num_key_value_heads`, `hidden_size / num_attention_heads`). Quantos pedidos de 2048 tokens cabem na memória que sobra na tua GPU?

In [ ]:
del modelo; torch.cuda.empty_cache()

## 2. Quantizar e comparar

Quantizamos com GPTQ 4 bits (e FP8, se a GPU suportar) e comparamos tamanho e exatidão no conjunto de teste do módulo 11.

In [ ]:
correr("quantizar.py", "--esquema", "w4a16")
for d in sorted((UC / "modelos").glob("*")):
    print(d.name, f"{sum(f.stat().st_size for f in d.glob('*.safetensors')) / 1e9:.2f} GB")
print("16 bits:", f"{sum(f.stat().st_size for f in MODELO.glob('*.safetensors')) / 1e9:.2f} GB")

A exatidão do modelo quantizado mede-se já com o vLLM na secção 3, porque o formato comprimido é pensado para ele.

## 3. Servir com vLLM

Arrancamos o vLLM em segundo plano e enviamos 200 pedidos em simultâneo pela API compatível com a da OpenAI.

In [ ]:
def arrancar_vllm(pasta, porta=8000):
    p = subprocess.Popen(["vllm", "serve", str(pasta), "--served-model-name", "reclamacoes", "--max-model-len", "2048",
                          "--gpu-memory-utilization", "0.85", "--port", str(porta)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    import httpx
    for _ in range(120):
        try:
            if httpx.get(f"http://localhost:{porta}/health", timeout=1).status_code == 200:
                print("vLLM pronto"); return p
        except httpx.HTTPError:
            time.sleep(2)
    p.kill(); raise RuntimeError("o vLLM não arrancou")

vllm = arrancar_vllm(UC / "modelos" / "reclamacoes-w4a16")

In [ ]:
import asyncio
from openai import AsyncOpenAI
from comum import SYSTEM, Extracao, ler_json, metricas

teste = [json.loads(l) for l in (M11 / "dados/teste_bruto.jsonl").read_text(encoding="utf-8").splitlines()]
cli = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="local")
esquema = {"type": "json_schema", "json_schema": {"name": "extracao", "schema": Extracao.model_json_schema()}}

async def um(x):
    r = await cli.chat.completions.create(model="reclamacoes", temperature=0, max_tokens=160, response_format=esquema,
                                          messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": x["texto"]}])
    return r.choices[0].message.content

t0 = time.perf_counter()
saidas = await asyncio.gather(*(um(x) for x in teste))
dur = time.perf_counter() - t0
m = metricas([ler_json(s) for s in saidas], [x["rotulo"] for x in teste])
print(f"{len(teste)} pedidos em {dur:.1f} s ({len(teste) / dur:.0f} pedidos/s)")
print({k: f"{v:.1%}" for k, v in m.items() if k != "n"})

Compara com a exatidão do modelo em 16 bits que mediste no módulo 11 (`resultados/` do M11). Se a perda for maior do que um ou dois pontos num campo, experimenta FP8 ou W8A16.

**Exercício 3.1.** Repete com `response_format` removido. Quantas respostas deixam de ser JSON válido? Que custo tem a geração guiada por esquema na latência?

## 4. A API de produção e o teste de carga

In [ ]:
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "servidor:app", "--port", "8080"], cwd=UC,
                       env={**os.environ, "MODELO_FALLBACK": ""}, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
time.sleep(4)
import httpx
print(httpx.get("http://localhost:8080/saude").json())
r = httpx.post("http://localhost:8080/classificar", headers={"X-API-Key": "dev"},
               json={"texto": "Fui cobrada duas vezes pela encomenda 77120. Se não resolverem vou ao Livro de Reclamações."})
print(r.json())
r = httpx.post("http://localhost:8080/classificar", headers={"X-API-Key": "dev"},
               json={"texto": "Fui cobrada duas vezes pela encomenda 77120.  Se não resolverem vou ao Livro de Reclamações."})
print("segunda vez:", r.json()["origem"], r.json()["latencia_ms"], "ms")

In [ ]:
correr("carga.py", "--nome", "w4a16", "--niveis", "1,8,32,64", "--n", "200")

Lê a tabela: a partir de que concorrência o p95 dispara? A exatidão mantém-se sob carga (deve: o modelo é o mesmo)?

**Exercício 4.1.** Baixa `LIMITE_POR_MINUTO` para 100 e repete o teste de carga. Que códigos HTTP aparecem? Como deve um cliente bem comportado reagir a um 429?

## 5. Monitorizar

In [ ]:
print(json.dumps(httpx.get("http://localhost:8080/metricas").json(), indent=1))

O painel (`uv run python painel.py`, em http://localhost:7860) mostra o mesmo ao longo do tempo. Um alerta de drift simples compara a distribuição das categorias de hoje com uma referência:

In [ ]:
import sqlite3
from collections import Counter

with sqlite3.connect(UC / "metricas.db") as con:
    hoje = Counter(c for (c,) in con.execute("SELECT categoria FROM pedidos WHERE categoria IS NOT NULL"))
ref = Counter(x["rotulo"]["categoria"] for x in teste)
tot_h, tot_r = sum(hoje.values()), sum(ref.values())
for cat in sorted(set(ref) | set(hoje)):
    ph, pr = hoje[cat] / tot_h, ref[cat] / tot_r
    alerta = "  ⚠️" if abs(ph - pr) > 0.10 else ""
    print(f"{cat:<20} referência {pr:5.1%}  hoje {ph:5.1%}{alerta}")

In [ ]:
api.terminate(); vllm.terminate()

**Exercício 5.1.** Acrescenta à API uma amostragem de 2 % dos pedidos para um juiz (o modelo grande do módulo 11 como referência) e mostra a concordância diária no painel. É a métrica de qualidade em produção que falta.

## Use case: API de produção para o modelo afinado

```bash
cd modulos/12-inferencia-eficiente-e-llmops/use-case
uv run python quantizar.py --esquema w4a16
MODELO=reclamacoes-w4a16 docker compose up --build
uv run python carga.py --nome w4a16 && uv run python painel.py
```

**Desafio:** faz deploy do vLLM no Modal ou num Space do Hugging Face com GPU, aponta a API para lá (`VLLM_URL`) e mede o cold start e o custo por 1000 pedidos ao longo de um dia de tráfego simulado.